# 4.24 — Anomaly & outlier detection
Anomaly & outlier detection turns unlabeled data into structure by choosing the right notion of similarity, compression, or surprise. Density, distance, and margin ideas become scores for identifying points that do not behave like the rest. Save a copy to Drive to edit.

In [ ]:

import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import SpectralBiclustering
from sklearn.datasets import load_digits, load_iris, make_blobs, make_moons
from sklearn.ensemble import IsolationForest
from sklearn.metrics import adjusted_rand_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KernelDensity, LocalOutlierFactor
from sklearn.preprocessing import MinMaxScaler, StandardScaler


def cluster_ladder():
    """D1..D5 clustering ladder of rising difficulty. Returns [(name, X, y_true, k), ...]."""
    rungs = []

    x1 = np.array([[0.0, 0.0], [0.3, 0.2], [3.0, 3.0], [3.2, 2.8], [0.1, 3.1], [0.2, 2.9]])
    y1 = np.array([0, 0, 1, 1, 2, 2])
    rungs.append(("D1 hand 3 clusters", x1, y1, 3))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.7, random_state=1)
    rungs.append(("D2 clean blobs", x2, y2, 3))

    x3, y3 = make_blobs(n_samples=240, centers=3, cluster_std=1.6, random_state=2)
    transform = np.array([[0.6, -0.6], [-0.4, 0.8]])
    x3 = x3 @ transform
    rungs.append(("D3 anisotropic + overlap", x3, y3, 3))

    iris = load_iris()
    rungs.append(("D4 Iris (real, 4-D)", iris.data, iris.target, 3))

    digits = load_digits()
    keep = np.isin(digits.target, [0, 1, 2, 3])
    rungs.append(("D5 digits 0-3 (real, 64-D)", digits.data[keep] / 16.0, digits.target[keep], 4))

    return rungs


def matrix_for_biclustering(X):
    X = np.asarray(X, dtype=float)
    return MinMaxScaler().fit_transform(X) + 0.001


def project_for_plot(X):
    X = np.asarray(X, dtype=float)
    if X.shape[1] == 2:
        return X
    centered = StandardScaler().fit_transform(X)
    u, s, vt = np.linalg.svd(centered, full_matrices=False)
    return centered @ vt[:2].T


def preview_rungs(rungs):
    for name, X, y, k in rungs:
        counts = dict(zip(*np.unique(y, return_counts=True)))
        print(name, "shape=", X.shape, "k=", k, "labels=", counts)
        print(np.round(X[:3, : min(5, X.shape[1])], 3))


def make_anomaly_ladder():
    rungs = []

    X1 = np.array([[0.0, 0.0], [0.2, 0.0], [0.0, 0.1], [4.0, 4.0]])
    y1 = np.array([0, 0, 0, 1])
    rungs.append(("D1 hand normals + one outlier", X1, y1, 2))

    for idx, (name, X, labels, k) in enumerate(cluster_ladder()[1:], start=2):
        rng = np.random.default_rng(240 + idx)
        X = np.asarray(X, dtype=float)
        n_outliers = max(8, int(0.08 * len(X)))
        lo = X.min(axis=0)
        hi = X.max(axis=0)
        span = np.maximum(hi - lo, 1.0)
        low_outliers = lo - rng.uniform(1.5, 2.5, size=(n_outliers // 2, X.shape[1])) * span
        high_outliers = hi + rng.uniform(1.5, 2.5, size=(n_outliers - n_outliers // 2, X.shape[1])) * span
        outliers = np.vstack([low_outliers, high_outliers])
        X_aug = np.vstack([X, outliers])
        y_aug = np.concatenate([np.zeros(len(X), dtype=int), np.ones(n_outliers, dtype=int)])
        rungs.append((name.replace("clusters", "normal groups") + " + synthetic outliers", X_aug, y_aug, 2))

    return rungs


def make_transaction_ladder():
    ladders = []
    base = [
        {"bread", "milk"},
        {"bread", "milk", "eggs"},
        {"bread", "eggs"},
        {"milk", "eggs"},
    ]
    labels = np.array([0, 0, 1, 1])
    ladders.append(("D1 four baskets", base, labels, 2))

    for name, X, y, k in cluster_ladder()[1:]:
        X = StandardScaler().fit_transform(X)
        transactions = []
        for row in X:
            items = set()
            for j, value in enumerate(row[: min(8, X.shape[1])]):
                if value <= -0.5:
                    items.add(f"f{j}_low")
                elif value >= 0.5:
                    items.add(f"f{j}_high")
                else:
                    items.add(f"f{j}_mid")
            transactions.append(items)
        ladders.append((name + " discretized transactions", transactions, y, k))

    return ladders


def print_table(rows, metric_name):
    print(f"{'rung':<38} {metric_name:>12}")
    for name, value in rows:
        print(f"{name:<38} {value:>12.3f}")


## The concept, built once (D1): center score
The lesson's distance score is $$s(x)=\lVert x-c\rVert_2.$$ We first estimate the center from normal points only, then score every point by distance from that center.

In [ ]:

def center_scores(X, normal_mask):
    center = X[normal_mask].mean(axis=0)
    scores = np.linalg.norm(X - center, axis=1)
    return center, scores

X_lesson = np.array([[0.0, 0.0], [0.2, 0.0], [0.0, 0.1], [4.0, 4.0]])
normal_mask = np.array([True, True, True, False])
center, scores = center_scores(X_lesson, normal_mask)
print("lesson center", np.round(center, 3))
assert np.allclose(np.round(center, 3), np.array([0.067, 0.033]))


## The concept, built once (D1): real detector scores
The reusable method fits real unsupervised detectors: Isolation Forest and Local Outlier Factor. Scores are oriented so larger means more anomalous, matching the lesson score.

In [ ]:

def method(X, contamination=0.1):
    X_scaled = StandardScaler().fit_transform(X)
    iso = IsolationForest(n_estimators=120, contamination=contamination, random_state=0)
    iso.fit(X_scaled)
    iso_scores = -iso.score_samples(X_scaled)
    neighbors = max(2, min(20, len(X_scaled) - 1))
    lof = LocalOutlierFactor(n_neighbors=neighbors, contamination=contamination)
    lof.fit_predict(X_scaled)
    lof_scores = -lof.negative_outlier_factor_
    combined = (iso_scores - iso_scores.mean()) / iso_scores.std()
    combined += (lof_scores - lof_scores.mean()) / lof_scores.std()
    threshold = np.quantile(combined, 1.0 - contamination)
    labels = (combined >= threshold).astype(int)
    return combined, labels

outlier_score = float(scores[-1])
normal_max = float(scores[normal_mask].max())
print("lesson outlier score", round(outlier_score, 3))
print("lesson largest normal score", round(normal_max, 3))
assert round(outlier_score, 3) == 5.586
assert round(normal_max, 3) == 0.137


## The dataset ladder
`cluster_ladder()` supplies normal structure; each rung gets seeded, far-away outliers. The metric is ROC-AUC of continuous outlier scores.

In [ ]:

rungs = make_anomaly_ladder()
preview_rungs(rungs)


## Run the same method across D1–D5
We fit without labels and use the hidden anomaly mask only to compute ROC-AUC afterward.

In [ ]:

rows = []
artifacts = []
for name, X, y_outlier, k in rungs:
    contamination = float(y_outlier.mean())
    scores, labels = method(X, contamination=contamination)
    auc = roc_auc_score(y_outlier, scores)
    rows.append((name, auc))
    artifacts.append((name, X, y_outlier, scores, labels))
print_table(rows, "ROC-AUC")


## Results visualization
The left panels show anomaly scores or score histograms, while the right curve shows ROC-AUC across the ladder.

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
flat_axes = axes.ravel()
for ax, artifact in zip(flat_axes, artifacts):
    name, X, y_outlier, scores, labels = artifact
    Z = project_for_plot(X)
    scatter = ax.scatter(Z[:, 0], Z[:, 1], c=scores, cmap="magma", s=18)
    ax.scatter(Z[labels == 1, 0], Z[labels == 1, 1], facecolors="none", edgecolors="cyan", s=55)
    ax.set_title(name.split("+")[0])
    ax.set_xlabel("view 1")
    ax.set_ylabel("view 2")
flat_axes[-1].plot(range(1, 6), [value for _, value in rows], marker="o")
flat_axes[-1].set_xticks(range(1, 6))
flat_axes[-1].set_xlabel("rung")
flat_axes[-1].set_ylabel("ROC-AUC")
flat_axes[-1].set_title("Outlier ranking vs. complexity")
plt.tight_layout()


## Pitfall on D5: contamination and scaling change flagged digits
Unscaled features and guessed contamination can flag false positives. The fix is to scale first and calibrate the threshold to an expected review budget or validation estimate.

In [ ]:

name, X_d5, y_d5, _ = rungs[-1]
raw_scores, raw_labels = method(X_d5 * np.linspace(1.0, 40.0, X_d5.shape[1]), contamination=0.20)
fixed_scores, fixed_labels = method(X_d5, contamination=float(y_d5.mean()))
raw_false_positive_rate = raw_labels[y_d5 == 0].mean()
fixed_false_positive_rate = fixed_labels[y_d5 == 0].mean()
raw_auc = roc_auc_score(y_d5, raw_scores)
fixed_auc = roc_auc_score(y_d5, fixed_scores)
print("wrong false-positive rate", round(raw_false_positive_rate, 3))
print("fixed false-positive rate", round(fixed_false_positive_rate, 3))
print("wrong ROC-AUC", round(raw_auc, 3))
print("fixed ROC-AUC", round(fixed_auc, 3))


## Evaluate it + Practice
- Compare the rung metric with a no-skill baseline: random row labels, random anomaly scores, one global density, or independence-only rules.
- Run a cheap sanity check: shuffle one key input and confirm the metric or discovered artifact degrades.
- Ablate the key idea: remove scaling, held-out scoring, bandwidth selection, or lift filtering and watch the metric drop.
- Inspect failure signals: unstable assignments, high training-only fit, score thresholds that flag whole subgroups, or rules with high support but lift near 1.
- Keep the hidden labels for evaluation only; the unsupervised method must not see them during fitting.

Practice prompts:
1. Change one hyperparameter near the default and predict which rung is most sensitive.


In [ ]:
# Your experiment for practice prompt 1 goes here.

2. Add a scaling or stability check and describe the before/after metric.

In [ ]:
# Your experiment for practice prompt 2 goes here.

3. Replace the D1 toy data with your own four-to-six point example and keep the asserts auditable.

In [ ]:
# Your experiment for practice prompt 3 goes here.